Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Cu(111)スラブ/水界面のモデリング

ASEとpfcc_extrasを使用して、Cu(111)スラブと水分子からなる固液界面構造を作成します。

このNotebookで実施する主な工程は以下のとおりです:
1. **Cu(111)スラブ構造の作成:** ASEの`fcc111`関数で6×8×6のスラブを構築。
2. **液体構造の作成:** pfcc_extrasの`LiquidGenerator`で水250分子を配置し、構造最適化。
3. **固液界面の作成:** スラブと液体を貼り合わせて界面構造を生成・最適化。

## Step 1: ライブラリのインポート

必要なパッケージをインポートします。

In [ ]:
from pathlib import Path

import numpy as np
from ase.build import fcc111
from ase import Atoms
from ase import units
from ase.data import atomic_masses
from ase.io import read, write
from ase.optimize import FIRE
from tqdm.notebook import tqdm

from pfcc_extras.liquidgenerator.liquid_generator import LiquidGenerator
from pfcc_extras.structure.ase_rdkit_converter import smiles_to_atoms
from pfcc_extras.structure.surface import makesurface
from pfcc_extras.visualize import show_gui
from pfcc_extras.structure.molecule import wrap_molecule

from pfp_api_client import ASECalculator, Estimator

## Step 1-2: 出力先フォルダの作成

In [ ]:
out_dir = "output/01_modeling"

out_dir = Path(out_dir)
out_dir.mkdir(exist_ok=True, parents=True)
print(f'出力先フォルダ {out_dir} が作成されました。')

## Step 2: Cuスラブ構造の作成

ASEの `fcc111` 関数を使って、Cu(111)面のスラブ構造を作成します。
サイズは6×8×6（288原子）で、上下に1 Åの真空層を設けます。

In [ ]:
# 111面でスラブ構造を作成する (上下に1Åずつの真空層が確保される)
slab = fcc111('Cu', size=(6, 8, 6), vacuum=1, orthogonal=True)

# 最下層のZ座標（最小のZ値）を取得し、その分だけZ軸のマイナス方向へ平行移動
z_min = slab.positions[:, 2].min()
slab.translate([0, 0, -z_min])

# 周期境界条件を設定する
slab.pbc = (True, True, True)

# 構造を保存する
slab_file = out_dir / 'cu_slab.cif'
slab.write(str(slab_file))
print(f'{slab_file} を保存しました')

In [ ]:
# 構造を可視化
show_gui(slab, representations=['ball+stick'])

## Step 3: 液体構造の作成

pfcc_extrasの`LiquidGenerator`を使って水分子250個の液体構造を生成します。

### Step 3-1: 溶液分子、密度、分子数の指定

In [ ]:
# 溶液分子を指定
mols = {
    'water': smiles_to_atoms('O'), # SMILESで水分子を指定する
}

# 密度を指定する
density = 1.0

# 溶液中の分子数
numbers = {
    'water': 250
}

In [ ]:
# スラブと溶媒の原子数をカウントする
num_liq = sum(len(mols[name]) * numbers[name] for name in mols)
print(f'Number of Atoms (Solid): {len(slab)}')
print(f'Number of Atoms (Liquid): {num_liq}')

### Step 3-2: 液体構造生成の条件パラメータ指定
`LiquidGenerator`を使って溶液の構造を作成します。 これは、分子同士の衝突の有無を判定し、繰り返し計算によりできる限り分子間の距離の離れた構造を生成するものです。

以下では繰り返し計算の回数と出力先のファイル名を指定してください。  

| パラメータ名 | 説明 |
| :--- | :--- |
| `epochs` | 分子衝突判定の繰り返し処理の回数。分子の複雑性によって推奨値は異なるが20-100程度。 |
| `rough_file` | ラフに作った構造を保存するファイル |

**note**: ラフに液体構造を生成するときはepochsを増やすより、後に最適化などで構造を整えた方が全体の計算時間は短縮できます。  
構造最適化が破綻しない程度の回数のepochsがあれば十分です。

In [ ]:
# パラメータ設定
epochs = 20

# 出力先の設定
rough_file = out_dir / 'liquid_structure.cif'

print(f'生成された構造は {rough_file} に保存されます。')

### Step 3-3: 構造生成の実行
`LiquidGenerator`を使用して、溶液部分の構造を作成します。

In [ ]:
composition = [mols[name] for name in mols for _ in range(numbers[name])]

liquid_system = Atoms()
for i in composition:
    liquid_system += i

# セルの形を決める
mass = sum(atomic_masses[i] for i in liquid_system.get_atomic_numbers())
axis_a = slab.cell[0]
axis_b = slab.cell[1]
normal_vec = np.cross(axis_a, axis_b)
area = np.linalg.norm(normal_vec)

target_vol = (mass / units.kg / density * 1e27)
unit_axis_c = slab.cell[2] / np.linalg.norm(slab.cell[2])
unit_vol = np.dot(normal_vec, unit_axis_c)
coef = target_vol / unit_vol
additional_axis_c = unit_axis_c * coef
cell = np.array([axis_a, axis_b, additional_axis_c])

cell

In [ ]:
# LiquidGeneratorのパラメータ
params = {
    "density": density,
    'cell': cell,
    'wall': True,
    "composition": composition,
    "init_structure_pos": "bottom",  # centerも指定可能
    "retain_init_xy": True
}

# LiquidGeneratorの実行
generator = LiquidGenerator('torch', **params)
solution = generator.run(epochs=epochs)

In [ ]:
show_gui(solution, representations=['ball+stick'])

### Step 3-4: 溶液構造の最適化

作成された液体構造を安定化させるため構造最適化を実施します。

| パラメータ名 | 説明 |
| :--- | :--- |
| `calc_mode` | PFPのcalc_mode。 |
| `model_version` | PFPのモデルバージョン |
| `fmax` | 計算の終了条件です。いずれかの原子に働く最大の力が`fmax` [eV/Å]以下になったときに計算を終了します。 |

**note**: 構造緩和後に300 K前後のシミュレーションの実施を予定しているため、`fmax`には 0.2 eV/Å 程度と比較的ゆるい最適化でも問題ありません。

In [ ]:
# PFP settings
calc_mode = 'PBE_PLUS_D3'
model_version = 'v8.0.0'

# Optimization setting
fmax = 0.2

# log setting
opt_file = out_dir / 'water_opt.xyz'

In [ ]:
# atomsの設定
opt_atoms = solution.copy()
opt_atoms.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))

# 最適化の実行
dyn = FIRE(opt_atoms)
dyn.run(fmax=fmax)

# 構造の保存
write(str(opt_file), opt_atoms)

In [ ]:
# 構造の可視化
show_gui(opt_atoms, representations=['ball+stick'])

## Step 4: 固液界面の作成
### Step 4-1: 固体構造と液体構造の張り合わせ
上記で作成した固体部分の構造と液体部分の構造を貼り合わせることにより、固液界面の構造を作成します。

固液界面を貼り合わせる際、境界付近で固体と液体の原子が近接しすぎるのを防ぐため、両者の間に隙間(マージン)を設けることを推奨しています。
マージンの値は、以下のコードで構造を生成・確認しながら最適なものを決定してください。

マージンを広くしすぎると液体の密度が変動してしまうため、大きすぎる値は避ける必要があります。
目安として、出力される各原子の最大力（fmax）が 10 eV/Å を下回る程度のマージンが推奨されます。最終的には構造最適化を行うため、原子同士が極端に重なっていなければ大丈夫です。

| パラメータ名 | 説明 |
| :--- | :--- |
| `upper_margin` | スラブの上方と溶液の下方の間のマージンの厚み [Å] |
| `lower_margin` | スラブの下方と溶液の上方の間のマージンの厚み [Å] |

In [ ]:
upper_margin = 2.5
lower_margin = 2.5
interface_file = out_dir / 'cu_water_interface.cif'
interface_opt_file = out_dir / 'cu_water_interface_opt.cif'

### Step 4-2: 固液界面構造の生成
固体部分の構造と液体部分の構造をファイルから読み込んで貼り合わせます。

In [ ]:
# 作成された液体構造の取得
slab = read(str(slab_file))
liquid_atoms = read(str(opt_file))
wrap_molecule(liquid_atoms)

# 1. スラブの本当の厚み（Z方向の最大値 - 最小値）を取得
slab_z_max = slab.positions[:, 2].max()
slab_z_min = slab.positions[:, 2].min()
slab_thickness = slab_z_max - slab_z_min

# 2. 溶液（水）の本当の厚みを取得
liquid_z_max = liquid_atoms.positions[:, 2].max()
liquid_z_min = liquid_atoms.positions[:, 2].min()
liquid_thickness = liquid_z_max - liquid_z_min

# 3. 溶液をスラブの上 + upper_margin の位置に移動させる
#    (水の一番下の原子が、スラブの一番上の原子 + margin の位置に来るようにする)
shift_z = (slab_z_max + upper_margin) - liquid_z_min
liquid_atoms.positions[:, 2] += shift_z

# 4. 貼り合わせる
interface = slab + liquid_atoms

# 5. セルの高さを「スラブの厚み + 水の厚み + 上下マージン」に厳密に設定する
new_cell_z = slab_thickness + liquid_thickness + upper_margin + lower_margin
interface.cell[2, 2] = new_cell_z

# 周期境界の適用 (wrap)
interface.wrap()

# 界面にかかる最大の力の計算
interface.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))
forces = interface.get_forces()
fmax = np.max(np.abs(forces))

# 構造の表示
print(f'fmax = {fmax}, この数値が10以下程度になるマージンを推奨します。')
show_gui(interface)

### Step 4-3: 固液界面構造の最適化

貼り合わせた界面構造を構造最適化して安定化させます。

In [ ]:
# Matlantis settings
calc_mode = 'PBE_PLUS_D3'
model_version = 'v8.0.0'

# Optimization setting
fmax = 0.2

opt_atoms = interface.copy()
opt_atoms.calc = ASECalculator(Estimator(calc_mode=calc_mode, model_version=model_version))

dyn = FIRE(opt_atoms)
dyn.run(fmax=fmax)

show_gui(opt_atoms, representations=['ball+stick'])

### Step 4-4: 固液界面構造の保存

In [ ]:
interface.write(str(interface_file))
opt_atoms.write(str(interface_opt_file))

## Next Step
これで、Cu(111)スラブ/水界面のモデリングが完了しました。
次の [02_equilibrium_npzt_md_ja.ipynb](./02_equilibrium_npzt_md_ja.ipynb) では、分子動力学シミュレーションを使って、作成した界面構造を375 KのNPzTアンサンブルで平衡化していきます。